# Retail Shopping Assistant – Evaluation Notebook

Loads all trained models and reproduces every metric and plot from the report.
Run top-to-bottom after completing the training steps.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from retail_assistant.config import load_config, abs_path
cfg = load_config()
print('Config loaded. Seed:', cfg['seed'])

Config loaded. Seed: 42


## 1. Load Data Splits

In [2]:
train_df = pd.read_csv(abs_path(cfg['paths']['train_csv'])).fillna('')
val_df   = pd.read_csv(abs_path(cfg['paths']['val_csv'])).fillna('')
test_df  = pd.read_csv(abs_path(cfg['paths']['test_csv'])).fillna('')

print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
print('\nCategory distribution (test):')
print(test_df['category'].value_counts())

fig, ax = plt.subplots(figsize=(10, 4))
test_df['category'].value_counts().plot.bar(ax=ax, color='steelblue')
ax.set_title('Test split – category distribution')
ax.set_xlabel('Category')
ax.set_ylabel('Count')
plt.tight_layout()
fig_dir = abs_path(cfg['paths']['figures_dir'])
fig_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_dir / 'category_distribution.png', dpi=120)
plt.show()
print('Saved category_distribution.png')

Train: 4234, Val: 529, Test: 530

Category distribution (test):
category
shoes         50
jewellery     50
topwear       50
watches       50
sandals       50
bottomwear    50
belts         50
eyewear       50
bags          50
socks         50
headwear      30
Name: count, dtype: int64


Saved category_distribution.png


## 2. Search Evaluation (Recall@k, MRR)

In [3]:
from retail_assistant.models.embedder import ProductIndex
from retail_assistant.data import product_to_text
import random

idx_path = abs_path(cfg['paths']['index_dir'])
assert (idx_path / 'emb.npy').exists(), 'Run: python -m train.build_index first'

use_faiss = cfg.get('embedder', {}).get('use_faiss', True)
index = ProductIndex.load(idx_path, cfg['embedder']['model_name'], use_faiss=use_faiss)
print(f'Index loaded: {len(index.df)} products, emb shape: {index.emb.shape}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4118.10it/s]

Index loaded: 5293 products, emb shape: (5293, 384)


In [4]:
random.seed(cfg['seed'])
np.random.seed(cfg['seed'])

sample = test_df.sample(min(200, len(test_df)), random_state=cfg['seed'])
K_VALUES = [1, 5, 10]
recall = {k: [] for k in K_VALUES}
rr_list = []

for _, row in sample.iterrows():
    query = product_to_text(row)
    hits = index.search(query, k=max(K_VALUES))
    pid = row.get('product_id')
    if pid and 'product_id' in hits.columns:
        hits = hits[hits['product_id'] != pid]
    cats = hits['category'].tolist()
    true_cat = str(row['category'])
    for k in K_VALUES:
        recall[k].append(float(true_cat in cats[:k]))
    rr = 0.0
    for rank, cat in enumerate(cats, 1):
        if cat == true_cat:
            rr = 1.0 / rank
            break
    rr_list.append(rr)

search_metrics = {'MRR': round(float(np.mean(rr_list)), 4)}
for k in K_VALUES:
    search_metrics[f'Recall@{k}'] = round(float(np.mean(recall[k])), 4)

print('Search metrics:')
for k, v in search_metrics.items():
    print(f'  {k}: {v}')

# Save
reports_dir = abs_path(cfg['paths']['reports_dir'])
reports_dir.mkdir(parents=True, exist_ok=True)
(reports_dir / 'search_results.json').write_text(json.dumps(search_metrics, indent=2))
print('Saved search_results.json')

Search metrics:
  MRR: 0.9762
  Recall@1: 0.965
  Recall@5: 0.995
  Recall@10: 1.0
Saved search_results.json


## 3. Content Generator Evaluation

In [5]:
from retail_assistant.models.content_gen import ContentGenerator
from retail_assistant.data import build_generation_prompt
from rouge_score import rouge_scorer as rs

gen_dir = abs_path(cfg['paths']['generator_dir'])
if gen_dir.exists():
    gen_ft = ContentGenerator(str(gen_dir))
    print('Loaded fine-tuned generator from', gen_dir)
else:
    gen_ft = None
    print('Fine-tuned generator not found – using zero-shot only')

gen_zs = ContentGenerator(cfg['generator']['base_model'])
print('Zero-shot generator loaded')

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 5258.31it/s]


[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loaded fine-tuned generator from E:\genai\retail-shopping-assistant\retail-shopping-assistant\models_out\content_gen


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 5434.74it/s]


[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Zero-shot generator loaded


In [6]:
scorer = rs.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
test_sample = test_df.head(50)  # cap for speed

refs, preds_zs, preds_ft = [], [], []
for _, r in test_sample.iterrows():
    prompt = 'Write a product description: ' + build_generation_prompt(r).split(': ', 1)[-1]
    refs.append(r['description'])
    preds_zs.append(gen_zs.generate(prompt, max_new_tokens=80))
    if gen_ft:
        preds_ft.append(gen_ft.generate(prompt, max_new_tokens=80))

def avg_rouge(preds, refs):
    agg = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    for p, r in zip(preds, refs):
        s = scorer.score(r, p)
        for k in agg: agg[k].append(s[k].fmeasure)
    return {k: round(float(np.mean(v)), 4) for k, v in agg.items()}

metrics_zs = avg_rouge(preds_zs, refs)
metrics_ft = avg_rouge(preds_ft, refs) if preds_ft else {}

print('Zero-shot ROUGE:', metrics_zs)
print('Fine-tuned ROUGE:', metrics_ft)

gen_results = {'zero_shot': metrics_zs, 'fine_tuned': metrics_ft}
(reports_dir / 'generator_results.json').write_text(json.dumps(gen_results, indent=2))
print('Saved generator_results.json')

Zero-shot ROUGE: {'rouge1': 0.3607, 'rouge2': 0.2837, 'rougeL': 0.3547}
Fine-tuned ROUGE: {'rouge1': 0.9096, 'rouge2': 0.8885, 'rougeL': 0.9096}
Saved generator_results.json


In [7]:
# ROUGE comparison bar chart
metrics_keys = ['rouge1', 'rouge2', 'rougeL']
x = np.arange(len(metrics_keys))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width/2, [metrics_zs.get(k,0) for k in metrics_keys], width, label='Zero-shot', color='#6366f1')
if metrics_ft:
    ax.bar(x + width/2, [metrics_ft.get(k,0) for k in metrics_keys], width, label='Fine-tuned', color='#10b981')
ax.set_xticks(x)
ax.set_xticklabels(metrics_keys)
ax.set_ylabel('F1 Score')
ax.set_title('Content Generator: ROUGE Comparison')
ax.legend()
plt.tight_layout()
plt.savefig(fig_dir / 'rouge_comparison.png', dpi=120)
plt.show()
print('Saved rouge_comparison.png')

Saved rouge_comparison.png


## 4. Image Classifier Evaluation

In [8]:
ckpt_path = abs_path(cfg['paths']['classifier_ckpt'])
if not ckpt_path.exists():
    print('Classifier checkpoint not found. Run: python -m train.train_classifier')
else:
    import torch
    from torch.utils.data import DataLoader
    from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
    from retail_assistant.models.classifier import CategoryClassifier, ProductImageDataset, get_transforms
    
    clf = CategoryClassifier(ckpt_path)
    classes = clf.classes
    c2i = {k: i for i, k in enumerate(classes)}
    img_dir = abs_path(cfg['paths']['image_dir'])
    image_size = cfg['classifier']['image_size']
    
    test_ds = ProductImageDataset(test_df, img_dir, c2i, get_transforms(image_size, False))
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)
    
    preds, trues = [], []
    clf.model.eval()
    with torch.no_grad():
        for x, y in test_loader:
            preds += clf.model(x.to(clf.device)).argmax(1).cpu().tolist()
            trues += y.tolist()
    
    acc = accuracy_score(trues, preds)
    mf1 = f1_score(trues, preds, average='macro', zero_division=0)
    print(f'Test Accuracy: {acc:.4f}  Macro F1: {mf1:.4f}')
    print(classification_report(trues, preds, target_names=classes, zero_division=0))
    
    # Confusion matrix
    cm = confusion_matrix(trues, preds)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes, cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title('Confusion Matrix (Test Set)')
    plt.tight_layout()
    plt.savefig(fig_dir / 'confusion_matrix_eval.png', dpi=120)
    plt.show()
    print('Saved confusion_matrix_eval.png')

Test Accuracy: 0.9642  Macro F1: 0.9642
              precision    recall  f1-score   support

        bags       0.96      1.00      0.98        50
       belts       1.00      0.98      0.99        50
  bottomwear       0.94      0.98      0.96        50
     eyewear       1.00      1.00      1.00        50
    headwear       1.00      0.93      0.97        30
   jewellery       0.98      0.94      0.96        50
     sandals       0.96      0.86      0.91        50
       shoes       0.86      0.98      0.92        50
       socks       1.00      0.94      0.97        50
     topwear       0.98      0.98      0.98        50
     watches       0.96      1.00      0.98        50

    accuracy                           0.96       530
   macro avg       0.97      0.96      0.96       530
weighted avg       0.97      0.96      0.96       530



Saved confusion_matrix_eval.png


## 5. Shopping Assistant – 30-query Hit-rate Eval

In [9]:
from retail_assistant.assistant import ShoppingAssistant

asst = ShoppingAssistant(index, generator=None, top_k=5, refuse_hallucination=True)

# Synthetic test queries: (query_text, expected_category)
TEST_QUERIES = [
    ('I need running shoes', 'shoes'),
    ('Show me black bags', 'bags'),
    ('What watches do you have?', 'watches'),
    ('Looking for sunglasses', 'eyewear'),
    ('I want a blue top', 'topwear'),
    ('Find me slim-fit bottomwear', 'bottomwear'),
    ('Show belts', 'belts'),
    ('Cotton socks please', 'socks'),
    ('Any jewellery under $50?', 'jewellery'),
    ('I need a cap or hat', 'headwear'),
    ('Red running shoes', 'shoes'),
    ('Leather bags under $100', 'bags'),
    ('Quartz watches', 'watches'),
    ('UV protection sunglasses', 'eyewear'),
    ('Formal topwear white', 'topwear'),
    ('Slim jeans', 'bottomwear'),
    ('Metal buckle belts', 'belts'),
    ('Ankle socks', 'socks'),
    ('Silver jewellery', 'jewellery'),
    ('Baseball cap', 'headwear'),
    ('Comfortable sneakers black', 'shoes'),
    ('Laptop bag padded', 'bags'),
    ('Smart watch with heart rate', 'watches'),
    ('Aviator sunglasses', 'eyewear'),
    ('Long sleeve shirt', 'topwear'),
    ('Denim trousers', 'bottomwear'),
    ('Genuine leather belt', 'belts'),
    ('Cushioned sports socks', 'socks'),
    ('Gold earrings', 'jewellery'),
    ('Sun hat for beach', 'headwear'),
]

hits_at_1, hits_at_5 = 0, 0
results_rows = []
for query, expected_cat in TEST_QUERIES:
    asst.reset()
    resp = asst.chat(query)
    products = resp.get('products', [])
    cats = [p.get('category', '') for p in products]
    h1 = int(expected_cat in cats[:1])
    h5 = int(expected_cat in cats[:5])
    hits_at_1 += h1
    hits_at_5 += h5
    results_rows.append({'query': query, 'expected': expected_cat, 'hit@1': h1, 'hit@5': h5,
                         'top_cat': cats[0] if cats else 'none'})

n = len(TEST_QUERIES)
print(f'Hit@1: {hits_at_1}/{n} = {hits_at_1/n:.3f}')
print(f'Hit@5: {hits_at_5}/{n} = {hits_at_5/n:.3f}')

asst_results = {'hit_rate_at_1': round(hits_at_1/n, 4), 'hit_rate_at_5': round(hits_at_5/n, 4),
                'n_queries': n, 'per_query': results_rows}
(reports_dir / 'assistant_eval.json').write_text(json.dumps(asst_results, indent=2))
print('Saved assistant_eval.json')

pd.DataFrame(results_rows)

Hit@1: 29/30 = 0.967
Hit@5: 30/30 = 1.000
Saved assistant_eval.json


,query,expected,hit@1,hit@5,top_cat
0,I need running shoes,shoes,1,1,shoes
1,Show me black bags,bags,1,1,bags
2,What watches do you have?,watches,1,1,watches
3,Looking for sunglasses,eyewear,1,1,eyewear
4,I want a blue top,topwear,1,1,topwear
5,Find me slim-fit bottomwear,bottomwear,1,1,bottomwear
6,Show belts,belts,1,1,belts
7,Cotton socks please,socks,1,1,socks
8,Any jewellery under $50?,jewellery,1,1,jewellery
9,I need a cap or hat,headwear,1,1,headwear


## 6. Error Analysis

In [10]:
# Top misclassifications from assistant
df_res = pd.DataFrame(results_rows)
missed = df_res[df_res['hit@5'] == 0]
print(f'Queries with no hit in top-5 ({len(missed)}/{n}):')
print(missed[['query', 'expected', 'top_cat']].to_string(index=False))

# Qualitative generator examples (top-10 best/worst by rouge1)
if preds_zs and refs:
    print('\n--- Zero-shot Generator Sample ---')
    for i in range(min(5, len(preds_zs))):
        print(f'\nRef: {refs[i][:100]}')
        print(f'ZS:  {preds_zs[i][:100]}')
        if preds_ft:
            print(f'FT:  {preds_ft[i][:100]}')

print('\nEvaluation complete!')

Queries with no hit in top-5 (0/30):
Empty DataFrame
Columns: [query, expected, top_cat]
Index: []

--- Zero-shot Generator Sample ---

Ref: Introducing the Catwalk Women Golden Wedges by Generic. This gold shoes combines style with comfort.
ZS:  The Catwalk Women Golden Wedges are made of gold.
FT:  Introducing the Catwalk Women Golden Wedges by Generic. This gold shoes combines style with comfort.

Ref: Introducing the Rreverie Silver & Black Earrings by Generic. This silver jewellery combines style wi
ZS:  The Rreverie Silver & Black Earrings are made of silver.
FT:  Introducing the Rreverie Silver & Black Earrings by Generic. This silver jewellery combines style wi

Ref: Introducing the Proline Men Orange Printed T-shirt by Generic. This orange topwear combines style wi
ZS:  Proline Men Orange Printed T-shirt.
FT:  Introducing the Proline Men Orange Printed T-shirt by Generic. This orange topwear combines style wi

Ref: Introducing the Q&Q Women Gold Dial Analog Watch by Generic. T

## 7. Summary Metrics Table

Run all cells above, then the table below auto-populates.

In [11]:
def safe_json(path):
    p = abs_path('reports') / path
    return json.loads(p.read_text()) if p.exists() else {}

sr = safe_json('search_results.json')
gr = safe_json('generator_results.json')
ar = safe_json('assistant_eval.json')
cr = safe_json('classifier_results_resnet18.json')
ce = safe_json('classifier_results_efficientnet_b0.json')

summary = pd.DataFrame([
    {'Module': 'Classifier (ResNet18)', 'Metric': 'Test Accuracy', 'Value': cr.get('test_accuracy', 'N/A')},
    {'Module': 'Classifier (ResNet18)', 'Metric': 'Macro F1', 'Value': cr.get('test_macro_f1', 'N/A')},
    {'Module': 'Classifier (EfficientNet-B0)', 'Metric': 'Test Accuracy', 'Value': ce.get('test_accuracy', 'N/A')},
    {'Module': 'Classifier (EfficientNet-B0)', 'Metric': 'Macro F1', 'Value': ce.get('test_macro_f1', 'N/A')},
    {'Module': 'Generator (Zero-shot)', 'Metric': 'ROUGE-L', 'Value': gr.get('zero_shot', {}).get('rougeL', 'N/A')},
    {'Module': 'Generator (Fine-tuned)', 'Metric': 'ROUGE-L', 'Value': gr.get('fine_tuned', {}).get('rougeL', 'N/A')},
    {'Module': 'Search', 'Metric': 'Recall@5', 'Value': sr.get('Recall@5', 'N/A')},
    {'Module': 'Search', 'Metric': 'MRR', 'Value': sr.get('MRR', 'N/A')},
    {'Module': 'Assistant', 'Metric': 'Hit@5', 'Value': ar.get('hit_rate_at_5', 'N/A')},
])
print(summary.to_string(index=False))
summary

                      Module        Metric     Value
       Classifier (ResNet18) Test Accuracy  0.964151
       Classifier (ResNet18)      Macro F1  0.964217
Classifier (EfficientNet-B0) Test Accuracy       N/A
Classifier (EfficientNet-B0)      Macro F1       N/A
       Generator (Zero-shot)       ROUGE-L    0.3547
      Generator (Fine-tuned)       ROUGE-L    0.9096
                      Search      Recall@5     0.995
                      Search           MRR    0.9762
                   Assistant         Hit@5       1.0


,Module,Metric,Value
0,Classifier (ResNet18),Test Accuracy,0.964151
1,Classifier (ResNet18),Macro F1,0.964217
2,Classifier (EfficientNet-B0),Test Accuracy,N/A
3,Classifier (EfficientNet-B0),Macro F1,N/A
4,Generator (Zero-shot),ROUGE-L,0.3547
5,Generator (Fine-tuned),ROUGE-L,0.9096
6,Search,Recall@5,0.995
7,Search,MRR,0.9762
8,Assistant,Hit@5,1.0
